In [1]:
import numpy as np
import pandas as pd
import re

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
N_PCA_DESC = 5

MATERIAL_COL = "Material"
SMILES_COL = "SMILES"
SYNTHESIS_COL = "synthesis from"

MATERIAL_COLS_IN_RAW = [
    "Epoxy Type",
    "Hardener Type",
    "Catalyst",
    "Additive"
]


# =========================
# CASCADE REPRESENTATION PIPELINE
# =========================

def build_cascade_representation(
    smiles_path,
    raw_data_path=None,
    out_material_descriptor_path="Material_Cascade_Descriptors_PCA5.xlsx",
    out_raw_merged_path="Raw_Data_with_Cascade_Descriptors.xlsx"
):
    """
    Cascade molecular representation pipeline.

    This function generates molecular descriptor representations for materials
    using a cascade representation strategy.

    If a material has a valid SMILES string, RDKit molecular descriptors are
    calculated directly from the SMILES.

    If a material does not have a SMILES string, the function uses the
    'synthesis from' column to identify precursor materials. The descriptor
    representation of the material is then constructed by averaging the
    descriptor vectors of its precursor materials through cascade tracing.

    After descriptor generation, invalid values are cleaned, missing values
    are imputed, and PCA is used to reduce the descriptor space to 5 components.
    """


    smiles_df = pd.read_excel(smiles_path)
    smiles_df.columns = smiles_df.columns.str.strip()

    for col in [MATERIAL_COL, SMILES_COL, SYNTHESIS_COL]:
        if col not in smiles_df.columns:
            smiles_df[col] = ""

    smiles_df[MATERIAL_COL] = smiles_df[MATERIAL_COL].fillna("").astype(str).str.strip()
    smiles_df[SMILES_COL] = smiles_df[SMILES_COL].fillna("").astype(str).str.strip()
    smiles_df[SYNTHESIS_COL] = smiles_df[SYNTHESIS_COL].fillna("").astype(str).str.strip()

    material_to_smiles = dict(zip(smiles_df[MATERIAL_COL], smiles_df[SMILES_COL]))
    material_to_synthesis = dict(zip(smiles_df[MATERIAL_COL], smiles_df[SYNTHESIS_COL]))


    descriptor_names = [name for name, func in Descriptors._descList]
    descriptor_calculator = MoleculeDescriptors.MolecularDescriptorCalculator(
        descriptor_names
    )

    def calculate_rdkit_descriptors(smiles):
        if not isinstance(smiles, str) or smiles.strip() == "":
            return None

        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            return None

        try:
            descriptor_vector = np.array(
                descriptor_calculator.CalcDescriptors(mol),
                dtype=float
            )
            return descriptor_vector
        except Exception:
            return None


    def split_components(text):

        if not isinstance(text, str) or text.strip() == "":
            return []

        parts = re.split(r"\s*(?:\+|&|,|;)\s*", text.strip())
        return [p.strip() for p in parts if p.strip()]


    cascade_cache = {}

    def cascade_material_representation(material_name):
        """
        Generate the descriptor representation of a material using
        the cascade representation strategy.

        Case 1:
        If the material has SMILES, calculate RDKit descriptors directly.

        Case 2:
        If the material does not have SMILES, use the 'synthesis from'
        field to identify precursor materials. The precursor descriptors
        are generated through cascade tracing and averaged to represent
        the target material.
        """

        material_name = str(material_name).strip()

        if material_name in cascade_cache:
            return cascade_cache[material_name]

        smiles = material_to_smiles.get(material_name, "")

        if smiles:
            descriptor_vector = calculate_rdkit_descriptors(smiles)

            if descriptor_vector is not None:
                cascade_cache[material_name] = descriptor_vector
                return descriptor_vector

        synthesis_text = material_to_synthesis.get(material_name, "")
        precursor_materials = split_components(synthesis_text)

        precursor_descriptors = []

        for precursor in precursor_materials:
            precursor_descriptor = cascade_material_representation(precursor)

            if precursor_descriptor is not None:
                precursor_descriptors.append(precursor_descriptor)

        if len(precursor_descriptors) > 0:
            precursor_matrix = np.vstack(precursor_descriptors)

            cascade_descriptor = np.nanmean(
                precursor_matrix,
                axis=0
            )

            cascade_cache[material_name] = cascade_descriptor
            return cascade_descriptor

        cascade_cache[material_name] = None
        return None


    all_materials = set(smiles_df[MATERIAL_COL].tolist())

    raw_df = None

    if raw_data_path is not None:
        raw_df = pd.read_excel(raw_data_path)
        raw_df.columns = raw_df.columns.str.strip()

        for col in MATERIAL_COLS_IN_RAW:
            if col not in raw_df.columns:
                continue

            raw_df[col] = raw_df[col].fillna("").astype(str).str.strip()

            for cell in raw_df[col].unique():
                components = split_components(cell)

                for component in components:
                    all_materials.add(component)


    material_names = []
    descriptor_rows = []

    failed_materials = []

    for material in sorted(all_materials):
        descriptor_vector = cascade_material_representation(material)

        if descriptor_vector is not None:
            material_names.append(material)
            descriptor_rows.append(descriptor_vector)
        else:
            failed_materials.append(material)

    if len(descriptor_rows) == 0:
        raise ValueError("No valid descriptors were generated.")

    descriptor_matrix = np.array(descriptor_rows, dtype=float)


    descriptor_matrix[~np.isfinite(descriptor_matrix)] = np.nan
    descriptor_matrix[np.abs(descriptor_matrix) > 1e12] = np.nan


    descriptor_imputer = SimpleImputer(strategy="mean")
    descriptor_matrix_imputed = descriptor_imputer.fit_transform(
        descriptor_matrix
    )


    descriptor_scaler = StandardScaler()
    descriptor_matrix_scaled = descriptor_scaler.fit_transform(
        descriptor_matrix_imputed
    )


    descriptor_pca = PCA(n_components=N_PCA_DESC, random_state=42)
    descriptor_matrix_pca = descriptor_pca.fit_transform(
        descriptor_matrix_scaled
    )

    pca_columns = [
        f"descriptor {i+1}" for i in range(N_PCA_DESC)
    ]

    material_descriptor_df = pd.DataFrame(
        descriptor_matrix_pca,
        columns=pca_columns
    )

    material_descriptor_df.insert(0, MATERIAL_COL, material_names)


    material_descriptor_df.to_excel(
        out_material_descriptor_path,
        index=False
    )

    print(f"Saved material cascade descriptors to: {out_material_descriptor_path}")

    print("\nExplained variance ratio of PCA components:")
    for i, var in enumerate(descriptor_pca.explained_variance_ratio_, start=1):
        print(f"PC{i}: {var:.4f}")

    if len(failed_materials) > 0:
        print("\nMaterials without valid cascade representation:")
        for m in failed_materials:
            print(m)


    if raw_df is not None:

        material_to_pca_vector = {
            row[MATERIAL_COL]: row[pca_columns].values.astype(float)
            for _, row in material_descriptor_df.iterrows()
        }

        def material_cell_to_cascade_vector(cell):
            components = split_components(cell)
            vectors = []

            for component in components:
                if component in material_to_pca_vector:
                    vectors.append(material_to_pca_vector[component])

            if len(vectors) == 0:
                return np.zeros(N_PCA_DESC)

            return np.mean(np.vstack(vectors), axis=0)

        for col in MATERIAL_COLS_IN_RAW:
            if col not in raw_df.columns:
                continue

            vectors = np.vstack(
                raw_df[col].apply(material_cell_to_cascade_vector).values
            )

            for i in range(N_PCA_DESC):
                raw_df[f"{col} - cascade descriptor {i+1}"] = vectors[:, i]

        raw_df.to_excel(
            out_raw_merged_path,
            index=False
        )

        print(f"\nSaved raw dataset with cascade descriptors to: {out_raw_merged_path}")

    return material_descriptor_df